## Trazabilidad BI (diagnóstico)

Esta sección documenta explícitamente qué outputs (D6_*.csv) alimentan cada componente de los dashboards DT_TEND y DT_SECT. Power BI actúa como capa de visualización; la lógica principal se materializa en estos CSV como evidencia reproducible. Estos outputs constituyen evidencia oficial para auditoría metodológica.

**Filtros (slicers)**
- **Año / Período (YM):** desde `D6_trend_por_mes.csv` (columna de período) + dimensión `dw_dim_fecha` (modelo BI).
- **Sector:** desde `D6_cruces_sector_moneda.csv` (columna de sector_lic), según modelo semántico vigente.
- **Moneda:** desde `D6_moneda_mapeo_std.csv` (armonizada; **sin conversión**).

**Tarjetas (KPIs)**
- Coberturas / proporciones diagnósticas: `D6_kpi_resumen.csv` (y/o por mes en `D6_trend_por_mes.csv`).
- Medianas (OC/LIC y LIC/OC): `D6_kpi_resumen.csv`.

**Gráficos**
- Tendencia temporal: `D6_trend_por_mes.csv`.
- Distribución OC por LIC: `D6_dist_oc_por_lic.csv`.
- Distribución LIC por OC: `D6_dist_lic_por_oc.csv`.
- Cruces por Sector y Moneda: `D6_cruces_sector_moneda.csv`.

**Tablas / Nota metodológica**
- Control de meses incluidos/excluidos: `D6_control_meses_lic.csv`, `D6_control_meses_oc.csv`.
- Alerta de calidad (moneda/oferta): `D6_alerta_moneda_oferta.csv` (si aplica).


# Evidencia administrativa LIC–OC (agregada) — Soporte DT_TEND / DT_SECT

**TFM ChileCompra Data Platform — Fase 6 (BI) — Evidencia académica (UCM)**

## 1 — Propósito

Este notebook genera evidencia **agregada y reproducible** para el análisis administrativo en Power BI, sin realizar cruce transaccional 1:1 LIC–OC.  
Los resultados alimentan las páginas **DT_TEND** (tendencia) y **DT_SECT** (sectores / dimensiones) del reporte.

> ⚠️ **Nota metodológica:** estas métricas son **diagnósticas** (cobertura, distribución, consistencia por período/moneda/sector).  
> No se infiere causalidad ni se fuerza vínculo transaccional LIC–OC.

## 2 — Fuente de datos

- **Motor:** PostgreSQL (DW local)
- **Capa consultada:** según SQL implementado (DW/SEM o STG, según corresponda a tus vistas)
- **Ventana temporal:** `PERIODO_INI` → `PERIODO_FIN` (variables de entorno o defaults)

## 3 — Definiciones mínimas

- **Cobertura (administrativa):** proporción de registros con/ sin señal asociada (según definiciones de los campos en tus vistas).
- **Tendencia:** serie temporal agregada por período (YM).
- **Sector / Moneda:** segmentación sin conversión monetaria (se analiza por moneda).

## 4 — Consumo en Power BI (mapa 1:1)

### Página **DT_TEND**
Usa (principalmente):
- `outputs/D6_kpi_resumen.csv`
- `outputs/D6_trend_por_mes.csv`
- `outputs/D6_control_meses_lic.csv`
- `outputs/D6_control_meses_oc.csv`
- `outputs/D6_oc_con_sin_lic_por_mes.csv`

### Página **DT_SECT**
Usa (principalmente):
- `outputs/D6_cruces_sector_moneda.csv`
- `outputs/D6_moneda_mapeo_std.csv`
- `outputs/D6_alerta_moneda_oferta.csv`

Distribuciones (si están visualizadas):
- `outputs/D6_dist_oc_por_lic.csv`
- `outputs/D6_dist_lic_por_oc.csv`

## 5 — Ejecución reproducible y outputs

Ejecuta este notebook desde:

`fase7_auditoria_BI/Evidencia_BI_vs_SQL/DT_TEND_y_DT_SECT/`

Genera outputs (Linux):

`fase7_auditoria_BI/Evidencia_BI_vs_SQL/DT_TEND_y_DT_SECT/outputs/`

Luego se copian manualmente a Windows para consumo por Power BI:

`C:\PowerBI\TFM\02_Cruce\...`


In [1]:
# 1 - SETUP: librerías, parámetros y conexión (PostgreSQL del stack Docker)
import os
import re
from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine, text

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

# 1.1 - Ventana temporal (ajusta si corresponde)
PERIODO_INI = os.getenv("TFM_PERIODO_INI", "2024-09")
PERIODO_FIN = os.getenv("TFM_PERIODO_FIN", "2025-10")

# 1.2 - Directorios (según solicitud)
BASE_DIR = Path.cwd()
OUT_DIR  = BASE_DIR / "outputs/02_Cruce" 
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("OK - OUT_DIR:", OUT_DIR)

# 1.3 - Conexión DB (usa env vars; evita hardcode en tribunal)
DB_USER = os.getenv("CHILE_DB_USER", "chile_user")
DB_PASS = os.getenv("CHILE_DB_PASS", "CHANGE_ME")
DB_HOST = os.getenv("CHILE_DB_HOST", "localhost")
DB_PORT = os.getenv("CHILE_DB_PORT", "5433")
DB_NAME = os.getenv("CHILE_DB_NAME", "chilecompra")

if DB_PASS == "":
    raise ValueError("Falta CHILE_DB_PASS. Define la variable de entorno antes de ejecutar.")

engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}")
print("OK - conexión preparada")


OK - OUT_DIR: /home/engineer/Documents/Proyectos/TFM/docs/evidence/fase7_auditoria_BI/Evidencia_BI_vs_SQL/DT_TEND_y_DT_SECT/outputs/02_Cruce
OK - conexión preparada


In [2]:
# 2 - HELPERS: utilidades de metadata y control de tablas
def parse_yyyymm_from_table(name: str) -> str | None:
    m = re.search(r"(\d{4})_(\d{2})$", name)
    if not m:
        return None
    return f"{m.group(1)}-{m.group(2)}"

def month_range(ini: str, fin: str):
    # ini/fin formato YYYY-MM
    y0, m0 = map(int, ini.split("-"))
    y1, m1 = map(int, fin.split("-"))
    months = []
    y, m = y0, m0
    while (y < y1) or (y == y1 and m <= m1):
        months.append(f"{y:04d}-{m:02d}")
        m += 1
        if m == 13:
            m = 1
            y += 1
    return months

def list_tables(prefix: str):
    q = text("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema='public'
      AND table_type='BASE TABLE'
      AND table_name LIKE :pat
    ORDER BY table_name;
    """)
    with engine.connect() as conn:
        df = pd.read_sql(q, conn, params={"pat": f"{prefix}%"})
    return df["table_name"].tolist()

def table_columns(table_name: str) -> set:
    q = text("""
    SELECT column_name
    FROM information_schema.columns
    WHERE table_schema='public' AND table_name=:t
    """)
    with engine.connect() as conn:
        df = pd.read_sql(q, conn, params={"t": table_name})
    return set(df["column_name"].tolist())

def has_cols(table_name: str, required: list[str]) -> bool:
    cols = table_columns(table_name)
    return all(c in cols for c in required)

def best_col(table_name: str, candidates: list[str]) -> str | None:
    cols = table_columns(table_name)
    for c in candidates:
        if c in cols:
            return c
    return None

LIC_TABLES = [t for t in list_tables("stg_lic_") if parse_yyyymm_from_table(t)]
OC_TABLES  = [t for t in list_tables("stg_oc_")  if parse_yyyymm_from_table(t)]

print("LIC tablas:", len(LIC_TABLES), "OC tablas:", len(OC_TABLES))
print("Ejemplo LIC:", LIC_TABLES[:3])
print("Ejemplo OC :", OC_TABLES[:3])

MESES = month_range(PERIODO_INI, PERIODO_FIN)
print("Meses objetivo:", MESES[0], "->", MESES[-1], "| n =", len(MESES))


LIC tablas: 14 OC tablas: 14
Ejemplo LIC: ['stg_lic_2024_09', 'stg_lic_2024_10', 'stg_lic_2024_11']
Ejemplo OC : ['stg_oc_2024_09', 'stg_oc_2024_10', 'stg_oc_2024_11']
Meses objetivo: 2024-09 -> 2025-10 | n = 14


In [3]:
# 3 - CONTROL DE MESES: detectar drift / tablas inválidas y dejar evidencia reproducible

# 3.1 - Requisitos mínimos para análisis 2.4 (STG)
# LIC: necesitamos un identificador de licitación y una moneda estable para filtro (codigomoneda o moneda_adquisicion)
LIC_ID_CANDIDATES = ["codigoexterno", "codigo", "id"]
LIC_SECTOR_CANDIDATES = ["sector", "sectordelestado", "sector_del_estado", "sectorestado", "sector_estado"]

LIC_REQ_BASE = ["codigomoneda", "moneda_adquisicion"]  # al menos uno debe existir (validación más abajo)

# OC: necesitamos identificador de OC y referencia a LIC
OC_REQ = ["codigo", "codigolicitacion"]  # y idealmente tipomonedaoc / sector

rows_lic = []
for t in LIC_TABLES:
    per = parse_yyyymm_from_table(t)
    if per not in MESES:
        continue

    lic_id = best_col(t, LIC_ID_CANDIDATES)
    sector = best_col(t, LIC_SECTOR_CANDIDATES)
    cols = table_columns(t)

    ok_moneda = ("codigomoneda" in cols) or ("moneda_adquisicion" in cols)
    ok = (lic_id is not None) and ok_moneda

    motivo = []
    if lic_id is None: motivo.append("sin_id_lic")
    if not ok_moneda:  motivo.append("sin_moneda")
    if sector is None: motivo.append("sin_sector")

    rows_lic.append({
        "periodo": per,
        "tabla": t,
        "incluida": bool(ok),
        "lic_id_col": lic_id or "",
        "sector_col": sector or "",
        "tiene_codigomoneda": "codigomoneda" in cols,
        "tiene_moneda_adquisicion": "moneda_adquisicion" in cols,
        "tiene_moneda_oferta": "moneda_de_la_oferta" in cols,
        "motivo_exclusion": ";".join(motivo) if not ok else ""
    })

rows_oc = []
for t in OC_TABLES:
    per = parse_yyyymm_from_table(t)
    if per not in MESES:
        continue

    cols = table_columns(t)
    ok = all(c in cols for c in OC_REQ)

    motivo = []
    if not ok: motivo.append("schema_invalido_minimo")
    # (se reporta disponibilidad de moneda/sector si existe)
    rows_oc.append({
        "periodo": per,
        "tabla": t,
        "incluida": bool(ok),
        "tiene_codigo": "codigo" in cols,
        "tiene_codigolicitacion": "codigolicitacion" in cols,
        "tiene_tipomonedaoc": "tipomonedaoc" in cols,
        "tiene_monedaitem": "monedaitem" in cols,
        "tiene_sector": "sector" in cols,
        "motivo_exclusion": ";".join(motivo) if not ok else ""
    })

df_ctl_lic = pd.DataFrame(rows_lic).sort_values(["periodo", "tabla"]).reset_index(drop=True)
df_ctl_oc  = pd.DataFrame(rows_oc ).sort_values(["periodo", "tabla"]).reset_index(drop=True)

df_ctl_lic.to_csv(OUT_DIR / "D6_control_meses_lic.csv", index=False, encoding="utf-8")
df_ctl_oc.to_csv(OUT_DIR / "D6_control_meses_oc.csv", index=False, encoding="utf-8")

print("OK -> D6_control_meses_lic.csv | incluidas =", df_ctl_lic["incluida"].sum(), "/", len(df_ctl_lic))
print("OK -> D6_control_meses_oc.csv  | incluidas =", df_ctl_oc["incluida"].sum(), "/", len(df_ctl_oc))

display(df_ctl_oc[df_ctl_oc["incluida"] == False].head(10))


OK -> D6_control_meses_lic.csv | incluidas = 13 / 14
OK -> D6_control_meses_oc.csv  | incluidas = 13 / 14


,periodo,tabla,incluida,tiene_codigo,tiene_codigolicitacion,tiene_tipomonedaoc,tiene_monedaitem,tiene_sector,motivo_exclusion
6,2025-03,stg_oc_2025_03,False,False,False,False,False,False,schema_invalido_minimo


In [4]:
# 4 - EXTRACCIÓN BASE: construir datasets LIC/OC a nivel 'cabecera administrativa' (distinct)
#     Nota: se trabaja con STG y con DISTINCT para evitar multiplicación por ítems.

# 4.1 - Selección dinámica de columnas LIC por tabla (id/sector)
lic_parts = []
for _, r in df_ctl_lic[df_ctl_lic["incluida"] == True].iterrows():
    t = r["tabla"]; per = r["periodo"]
    lic_id = r["lic_id_col"]
    sector = r["sector_col"]

    # moneda base: priorizar codigomoneda (estable); si no existe, dejamos NULL y usamos moneda_adquisicion
    cols = table_columns(t)
    has_code = "codigomoneda" in cols
    has_adq  = "moneda_adquisicion" in cols
    has_ofe  = "moneda_de_la_oferta" in cols

    lic_parts.append(f"""
        SELECT DISTINCT
          '{per}' AS periodo,
          NULLIF(TRIM({lic_id}), '') AS lic_id,
          {('NULLIF(TRIM(' + sector + "), '')" ) if sector else "NULL"} AS sector_lic,
          {"NULLIF(TRIM(codigomoneda), '')" if has_code else "NULL"} AS codigomoneda,
          {"NULLIF(TRIM(moneda_adquisicion), '')" if has_adq else "NULL"} AS moneda_adquisicion,
          {"NULLIF(TRIM(moneda_de_la_oferta), '')" if has_ofe else "NULL"} AS moneda_de_la_oferta
        FROM public.{t}
        WHERE {lic_id} IS NOT NULL AND TRIM({lic_id}) <> ''
    """)

sql_lic = "\nUNION ALL\n".join(lic_parts)

# 4.2 - Selección OC por tabla (siempre codigo + codigolicitacion)
oc_parts = []
for _, r in df_ctl_oc[df_ctl_oc["incluida"] == True].iterrows():
    t = r["tabla"]; per = r["periodo"]
    cols = table_columns(t)
    has_moc = "tipomonedaoc" in cols
    has_mit = "monedaitem" in cols
    has_sec = "sector" in cols

    oc_parts.append(f"""
        SELECT DISTINCT
          '{per}' AS periodo,
          NULLIF(TRIM(codigo), '') AS oc_id,
          NULLIF(TRIM(codigolicitacion), '') AS lic_ref,
          {"NULLIF(TRIM(sector), '')" if has_sec else "NULL"} AS sector_oc,
          {"NULLIF(TRIM(tipomonedaoc), '')" if has_moc else "NULL"} AS tipomonedaoc,
          {"NULLIF(TRIM(monedaitem), '')" if has_mit else "NULL"} AS monedaitem
        FROM public.{t}
        WHERE codigo IS NOT NULL AND TRIM(codigo) <> ''
    """)

sql_oc = "\nUNION ALL\n".join(oc_parts)

with engine.connect() as conn:
    df_lic = pd.read_sql(text(sql_lic), conn)
    df_oc  = pd.read_sql(text(sql_oc), conn)

print("LIC rows:", len(df_lic), "| OC rows:", len(df_oc))
print("LIC meses:", sorted(df_lic['periodo'].unique())[:3], "...", sorted(df_lic['periodo'].unique())[-3:])
print("OC  meses:", sorted(df_oc['periodo'].unique())[:3], "...", sorted(df_oc['periodo'].unique())[-3:])

display(df_lic.head(3))
display(df_oc.head(3))


LIC rows: 157148 | OC rows: 2070139
LIC meses: ['2024-09', '2024-10', '2024-11'] ... ['2025-08', '2025-09', '2025-10']
OC  meses: ['2024-09', '2024-10', '2024-11'] ... ['2025-08', '2025-09', '2025-10']


,periodo,lic_id,sector_lic,codigomoneda,moneda_adquisicion,moneda_de_la_oferta
0,2024-09,1380-290-LE24,Salud,CLP,Peso Chileno,Peso Chileno
1,2024-09,3017-197-R124,Municipalidades,CLP,Peso Chileno,Peso Chileno
2,2024-09,1057979-39-I224,Salud,CLP,Peso Chileno,Peso Chileno


,periodo,oc_id,lic_ref,sector_oc,tipomonedaoc,monedaitem
0,2025-02,0-5-CM25,None,Municipalidades,CLP,CLP
1,2025-02,1000813-27-AG25,None,FFAA,CLP,CLP
2,2025-02,1000813-28-AG25,None,FFAA,CLP,CLP


In [5]:
# 5 - MONEDA_STD: armonización semántica (sin conversión de montos)

# 5.1 - Mapas
# NOTA METODOLÓGICA (D6 - DT_TEND / DT_SECT)
# ------------------------------------------------------------
# Este mapeo se utiliza EXCLUSIVAMENTE para el DT_TEND / DT_SECT (Evidencia administrativa (agregada)).
# Su propósito es armonizar etiquetas de moneda entre dominios LIC y OC para
# permitir filtrado consistente en Power BI.
#
# IMPORTANTE:
# - NO se realiza conversión monetaria.
# - NO se agregan montos entre monedas distintas.
# - La categoría 'REVISAR' representa un valor entregado por el sistema fuente
#   que indica moneda no clasificable / inconsistente (alerta de calidad).
#
# Equivalencias relevantes:
# - Unidad de Fomento (UF) ↔ CLF (ISO 4217)
# - Peso Chileno ↔ CLP
# - Dólar ↔ USD
# - Euro ↔ EUR
# - UTM ↔ UTM
# ------------------------------------------------------------

TEXT_TO_CODE = {
    # CLP
    "PESO CHILENO": "CLP",
    "PESOS CHILENOS": "CLP",
    "CLP": "CLP",

    # USD
    "DOLAR": "USD",
    "DÓLAR": "USD",
    "USD": "USD",

    # EUR
    "EURO": "EUR",
    "EUR": "EUR",

    # CLF / UF
    "UNIDAD DE FOMENTO": "CLF",
    "UF": "CLF",
    "U.F.": "CLF",
    "U F": "CLF",
    "CLF": "CLF",

    # UTM
    "UTM": "UTM",
    "UNIDAD TRIBUTARIA MENSUAL": "UTM",

    # calidad de dato
    "MONEDA REVISAR": "REVISAR",
    "REVISAR": "REVISAR",
}

VALID_CODES = {"CLP", "USD", "CLF", "UTM", "EUR"}

def norm_code(x):
    if x is None:
        return None
    x = str(x).strip().upper()
    return x if x in VALID_CODES else ("UNKNOWN" if x != "" else None)

def lic_moneda_std(row):
    # Prioridad: código estable; fallback: moneda_adquisicion textual
    c = norm_code(row.get("codigomoneda"))
    if c and c != "UNKNOWN":
        return c
    m = row.get("moneda_adquisicion")
    if m is None or str(m).strip() == "":
        return "UNKNOWN"
    return TEXT_TO_CODE.get(str(m).strip(), "UNKNOWN")

df_lic["moneda_std"] = df_lic.apply(lic_moneda_std, axis=1)
df_oc["moneda_std"]  = df_oc["tipomonedaoc"].apply(norm_code)

# 5.2 - Tabla de mapeo (evidencia)
df_map = (
    df_lic[["codigomoneda","moneda_adquisicion","moneda_std"]]
    .drop_duplicates()
    .sort_values(["moneda_std","codigomoneda","moneda_adquisicion"])
    .reset_index(drop=True)
)

df_map.to_csv(OUT_DIR / "D6_moneda_mapeo_std.csv", index=False, encoding="utf-8")
print("OK -> D6_moneda_mapeo_std.csv | filas =", len(df_map))

display(df_map.head(15))

OK -> D6_moneda_mapeo_std.csv | filas = 5


,codigomoneda,moneda_adquisicion,moneda_std
0,CLF,Unidad de Fomento,CLF
1,CLP,Peso Chileno,CLP
2,EUR,Euro,EUR
3,USD,Dolar,USD
4,UTM,Moneda revisar,UTM


In [6]:
# 6 - CRUCE ADMINISTRATIVO: métricas base para DT_TEND / DT_SECT

# 6.1 - LIC con OC (por código informado en OC)
# Regla: una LIC puede tener 0..N OC; una OC referencia 0/1 LIC (se valida aparte)
df_oc_ref = df_oc.copy()
df_oc_ref["lic_ref"] = df_oc_ref["lic_ref"].astype("string")

# join simple por lic_id = lic_ref (ambos administrativos)
df_join = df_lic.merge(
    df_oc_ref[["periodo","oc_id","lic_ref","moneda_std","sector_oc"]],
    left_on=["periodo","lic_id"],
    right_on=["periodo","lic_ref"],
    how="left",
    suffixes=("_lic","_oc")
)

# 6.2 - KPIs globales (universo analizado)
lic_total = df_lic["lic_id"].nunique()
lic_con_oc = df_join[df_join["oc_id"].notna()]["lic_id"].nunique()
pct_lic_con_oc = (100.0 * lic_con_oc / lic_total) if lic_total else 0.0

oc_total = df_oc["oc_id"].nunique()
oc_con_lic = df_oc[df_oc["lic_ref"].notna() & (df_oc["lic_ref"].astype(str).str.strip().str.upper() != "")]["oc_id"].nunique()
pct_oc_con_lic = (100.0 * oc_con_lic / oc_total) if oc_total else 0.0

# 6.3 - Mediana OC por LIC (solo LIC con OC)
oc_por_lic = (
    df_oc[df_oc["lic_ref"].notna() & (df_oc["lic_ref"].astype(str).str.strip().str.upper() != "")]
    .groupby("lic_ref")["oc_id"].nunique()
)
mediana_oc_por_lic = float(oc_por_lic.median()) if len(oc_por_lic) else float("nan")

# 6.4 - Mediana LIC por OC (integridad; debería ser <=1)
lic_por_oc = (
    df_oc.groupby("oc_id")["lic_ref"]
    .apply(lambda s: s.dropna().astype(str).str.strip().str.upper().replace("", pd.NA).dropna().nunique())
)
mediana_lic_por_oc = float(lic_por_oc.median()) if len(lic_por_oc) else float("nan")

df_kpi = pd.DataFrame([{
    "periodo_ini": PERIODO_INI,
    "periodo_fin": PERIODO_FIN,
    "lic_distintas": int(lic_total),
    "lic_con_oc": int(lic_con_oc),
    "pct_lic_con_oc": round(pct_lic_con_oc, 2),
    "oc_distintas": int(oc_total),
    "oc_con_lic": int(oc_con_lic),
    "pct_oc_con_lic": round(pct_oc_con_lic, 2),
    "mediana_oc_por_lic": mediana_oc_por_lic,
    "mediana_lic_por_oc": mediana_lic_por_oc,
}])

df_kpi.to_csv(OUT_DIR / "D6_kpi_resumen.csv", index=False, encoding="utf-8")
print("OK -> D6_kpi_resumen.csv")
display(df_kpi)

OK -> D6_kpi_resumen.csv


,periodo_ini,periodo_fin,lic_distintas,lic_con_oc,pct_lic_con_oc,oc_distintas,oc_con_lic,pct_oc_con_lic,mediana_oc_por_lic,mediana_lic_por_oc
0,2024-09,2025-10,136119,13716,10.08,2070139,786669,38.0,1.0,0.0


In [7]:
# 7 - TENDENCIA TEMPORAL (por mes): base para gráfico de líneas y tabla resumen

lic_mes = df_lic.groupby("periodo")["lic_id"].nunique().rename("lic_total")
lic_con_oc_mes = (
    df_join[df_join["oc_id"].notna()]
    .groupby("periodo")["lic_id"].nunique()
    .rename("lic_con_oc")
)

oc_mes = df_oc.groupby("periodo")["oc_id"].nunique().rename("oc_total")
oc_con_lic_mes = (
    df_oc[df_oc["lic_ref"].notna() & (df_oc["lic_ref"].astype(str).str.strip().str.upper() != "")]
    .groupby("periodo")["oc_id"].nunique()
    .rename("oc_con_lic")
)

df_trend = (
    pd.concat([lic_mes, lic_con_oc_mes, oc_mes, oc_con_lic_mes], axis=1)
    .reset_index()
    .rename(columns={"index":"periodo"})
    .fillna(0)
)

df_trend["pct_lic_con_oc"] = df_trend.apply(lambda r: round(100*r["lic_con_oc"]/r["lic_total"], 3) if r["lic_total"] else 0.0, axis=1)
df_trend["pct_oc_con_lic"] = df_trend.apply(lambda r: round(100*r["oc_con_lic"]/r["oc_total"], 3) if r["oc_total"] else 0.0, axis=1)

# Asegurar presencia de meses objetivo (aunque estén excluidos)
df_trend = (
    pd.DataFrame({"periodo": MESES})
    .merge(df_trend, on="periodo", how="left")
    .fillna(0)
)

df_trend.to_csv(OUT_DIR / "D6_trend_por_mes.csv", index=False, encoding="utf-8")
print("OK -> D6_trend_por_mes.csv | filas =", len(df_trend))
display(df_trend.head(12))


OK -> D6_trend_por_mes.csv | filas = 14


,periodo,lic_total,lic_con_oc,oc_total,oc_con_lic,pct_lic_con_oc,pct_oc_con_lic
0,2024-09,11496.0,1395.0,152598.0,57813.0,12.135,37.886
1,2024-10,15410.0,2457.0,192514.0,71010.0,15.944,36.886
2,2024-11,15068.0,2251.0,180691.0,67944.0,14.939,37.602
3,2024-12,8230.0,1569.0,156428.0,66340.0,19.064,42.409
4,2025-01,0.0,0.0,139821.0,61434.0,0.000,43.938
5,2025-02,8481.0,693.0,126438.0,49157.0,8.171,38.878
6,2025-03,9325.0,0.0,0.0,0.0,0.000,0.000
7,2025-04,10357.0,803.0,161980.0,58492.0,7.753,36.111
8,2025-05,9654.0,614.0,157037.0,58134.0,6.360,37.019
9,2025-06,9642.0,752.0,155481.0,57970.0,7.799,37.284


In [8]:
# 8 - DISTRIBUCIONES (histogramas): OC por LIC y LIC por OC

# 8.1 - OC por LIC (solo lic_ref no vacío)
oc_por_lic = (
    df_oc[df_oc["lic_ref"].notna() & (df_oc["lic_ref"].astype(str).str.strip().str.upper() != "")]
    .groupby("lic_ref")["oc_id"].nunique()
    .rename("n_oc")
    .reset_index()
)

# bins discretos (1,2,3,4,5+)
def bucket_oc(n):
    if n >= 5:
        return "5+"
    return str(int(n))

oc_por_lic["bucket"] = oc_por_lic["n_oc"].apply(bucket_oc)

df_dist_oc_por_lic = (
    oc_por_lic.groupby("bucket")["lic_ref"].nunique()
    .rename("frecuencia_lic")
    .reset_index()
    .rename(columns={"lic_ref":"lic_distintas"})
)
# ordenar buckets
order = ["1","2","3","4","5+"]
df_dist_oc_por_lic["bucket"] = pd.Categorical(df_dist_oc_por_lic["bucket"], categories=order, ordered=True)
df_dist_oc_por_lic = df_dist_oc_por_lic.sort_values("bucket").reset_index(drop=True)

df_dist_oc_por_lic.to_csv(OUT_DIR / "D6_dist_oc_por_lic.csv", index=False, encoding="utf-8")
print("OK -> D6_dist_oc_por_lic.csv")
display(df_dist_oc_por_lic)

# 8.2 - LIC por OC (integridad normativa): debería concentrarse en 0 y 1
lic_por_oc = (
    df_oc.groupby("oc_id")["lic_ref"]
    .apply(lambda s: s.dropna().astype(str).str.strip().str.upper().replace("", pd.NA).dropna().nunique())
    .rename("n_lic")
    .reset_index()
)

def bucket_lic(n):
    if n >= 5:
        return "5+"
    return str(int(n))

lic_por_oc["bucket"] = lic_por_oc["n_lic"].apply(bucket_lic)

df_dist_lic_por_oc = (
    lic_por_oc.groupby("bucket")["oc_id"].nunique()
    .rename("frecuencia_oc")
    .reset_index()
    .rename(columns={"oc_id":"oc_distintas"})
)
order2 = ["0","1","2","3","4","5+"]
df_dist_lic_por_oc["bucket"] = pd.Categorical(df_dist_lic_por_oc["bucket"], categories=order2, ordered=True)
df_dist_lic_por_oc = df_dist_lic_por_oc.sort_values("bucket").reset_index(drop=True)

df_dist_lic_por_oc.to_csv(OUT_DIR / "D6_dist_lic_por_oc.csv", index=False, encoding="utf-8")
print("OK -> D6_dist_lic_por_oc.csv")
display(df_dist_lic_por_oc)

# 8.3 - OC con/sin LIC por mes (tarjeta o gráfico simple)
df_oc_flag_mes = (
    df_oc.assign(tiene_lic=df_oc["lic_ref"].notna() & (df_oc["lic_ref"].astype(str).str.strip().str.upper() != ""))
    .groupby("periodo", as_index=False)
    .agg(
        oc_total=("oc_id","nunique"),
        oc_con_lic=("tiene_lic","sum")
    )
)
df_oc_flag_mes["oc_sin_lic"] = df_oc_flag_mes["oc_total"] - df_oc_flag_mes["oc_con_lic"]
df_oc_flag_mes["pct_oc_con_lic"] = df_oc_flag_mes.apply(lambda r: round(100*r["oc_con_lic"]/r["oc_total"], 3) if r["oc_total"] else 0.0, axis=1)

df_oc_flag_mes = pd.DataFrame({"periodo": MESES}).merge(df_oc_flag_mes, on="periodo", how="left").fillna(0)

df_oc_flag_mes.to_csv(OUT_DIR / "D6_oc_con_sin_lic_por_mes.csv", index=False, encoding="utf-8")
print("OK -> D6_oc_con_sin_lic_por_mes.csv")


OK -> D6_dist_oc_por_lic.csv


,bucket,frecuencia_lic
0,1,77229
1,2,13222
2,3,7056
3,4,5488
4,5+,35418


OK -> D6_dist_lic_por_oc.csv


,bucket,frecuencia_oc
0,0,1283470
1,1,786669


OK -> D6_oc_con_sin_lic_por_mes.csv


In [9]:
# 9 - CRUCE POR SECTOR Y MONEDA (base para gráfico de barras apiladas)

# Definición: se reporta universo LIC (por sector_lic y moneda_std),
# y cuántas de esas LIC tienen al menos 1 OC asociada (por código).
lic_base = df_lic.copy()
lic_base["sector_lic"] = lic_base["sector_lic"].fillna("(Sin sector)")
lic_base["moneda_std"] = lic_base["moneda_std"].fillna("UNKNOWN")

lic_con_oc_flag = (
    df_join.assign(tiene_oc=df_join["oc_id"].notna())
    .groupby(["periodo","lic_id"], as_index=False)["tiene_oc"].max()
)

lic_base2 = lic_base.merge(lic_con_oc_flag, on=["periodo","lic_id"], how="left")
lic_base2["tiene_oc"] = lic_base2["tiene_oc"].fillna(False)

df_sec_mon = (
    lic_base2.groupby(["sector_lic","moneda_std"], as_index=False)
    .agg(
        lic_total=("lic_id","nunique"),
        lic_con_oc=("tiene_oc","sum")
    )
)
df_sec_mon["pct_lic_con_oc"] = df_sec_mon.apply(lambda r: round(100*r["lic_con_oc"]/r["lic_total"], 3) if r["lic_total"] else 0.0, axis=1)

df_sec_mon.to_csv(OUT_DIR / "D6_cruces_sector_moneda.csv", index=False, encoding="utf-8")
print("OK -> D6_cruces_sector_moneda.csv")
display(df_sec_mon.sort_values(["lic_total"], ascending=False).head(15))


OK -> D6_cruces_sector_moneda.csv


,sector_lic,moneda_std,lic_total,lic_con_oc,pct_lic_con_oc
17,Municipalidades,CLP,58090,8598,14.801
29,Salud,CLP,30451,1214,3.987
8,"Gob. Central, Universidades",CLP,20154,2230,11.065
4,FFAA,CLP,11202,1780,15.890
21,Obras Públicas,CLP,6385,494,7.737
24,Otros,CLP,4654,547,11.753
1,(Sin sector),CLP,1547,166,10.730
13,Legislativo y judicial,CLP,733,47,6.412
10,"Gob. Central, Universidades",USD,515,33,6.408
7,"Gob. Central, Universidades",CLF,437,13,2.975


In [10]:
# 10 - ALERTA DE CALIDAD: 'moneda_de_la_oferta' (NO usar como slicer monetario principal)

if "moneda_de_la_oferta" in df_lic.columns:
    df_alert = (
        df_lic.assign(oferta=df_lic["moneda_de_la_oferta"].fillna(""))
        .groupby("periodo", as_index=False)
        .agg(
            lic_total=("lic_id","nunique"),
            n_oferta_moneda_revisar=("oferta", lambda s: (s.astype(str).str.strip().str.upper() == "Moneda revisar").sum()),
            n_oferta_vacia=("oferta", lambda s: (s.astype(str).str.strip().str.upper() == "").sum()),
        )
    )
    df_alert["pct_oferta_moneda_revisar"] = df_alert.apply(lambda r: round(100*r["n_oferta_moneda_revisar"]/r["lic_total"], 3) if r["lic_total"] else 0.0, axis=1)
    df_alert = pd.DataFrame({"periodo": MESES}).merge(df_alert, on="periodo", how="left").fillna(0)

    df_alert.to_csv(OUT_DIR / "D6_alerta_moneda_oferta.csv", index=False, encoding="utf-8")
    print("OK -> D6_alerta_moneda_oferta.csv")
    display(df_alert.head(12))
else:
    print("INFO: No existe columna moneda_de_la_oferta en df_lic (según tablas incluidas).")


OK -> D6_alerta_moneda_oferta.csv


,periodo,lic_total,n_oferta_moneda_revisar,n_oferta_vacia,pct_oferta_moneda_revisar
0,2024-09,11496.0,0.0,0.0,0.0
1,2024-10,15410.0,0.0,0.0,0.0
2,2024-11,15068.0,0.0,0.0,0.0
3,2024-12,8230.0,0.0,0.0,0.0
4,2025-01,0.0,0.0,0.0,0.0
5,2025-02,8481.0,0.0,0.0,0.0
6,2025-03,9325.0,0.0,0.0,0.0
7,2025-04,10357.0,0.0,0.0,0.0
8,2025-05,9654.0,0.0,0.0,0.0
9,2025-06,9642.0,0.0,0.0,0.0


## 6 — Validación mínima (reproducibilidad)

Checks mínimos para defensa:
- existencia de dataframes esperados
- shapes
- columnas clave (periodo/moneda/sector si aplica)


In [11]:
# 6.1 - Validación mínima (prints rápidos)
dfs = {
    "D6_control_meses_lic": df_ctl_lic,
    "D6_control_meses_oc": df_ctl_oc,
    "D6_moneda_mapeo_std": df_map,
    "D6_kpi_resumen": df_kpi,
    "D6_trend_por_mes": df_trend,
    "D6_dist_oc_por_lic": df_dist_oc_por_lic,
    "D6_dist_lic_por_oc": df_dist_lic_por_oc,
    "D6_oc_con_sin_lic_por_mes": df_oc_flag_mes,
    "D6_cruces_sector_moneda": df_sec_mon,
    "D6_alerta_moneda_oferta": df_alert,
}

for k, df in dfs.items():
    print(f"== {k} ==")
    print("shape:", df.shape)
    print("cols:", list(df.columns)[:20])
    print(df.head(3))
    print("-"*60)


== D6_control_meses_lic ==
shape: (14, 9)
cols: ['periodo', 'tabla', 'incluida', 'lic_id_col', 'sector_col', 'tiene_codigomoneda', 'tiene_moneda_adquisicion', 'tiene_moneda_oferta', 'motivo_exclusion']
   periodo            tabla  incluida     lic_id_col sector_col  tiene_codigomoneda  tiene_moneda_adquisicion  tiene_moneda_oferta motivo_exclusion
0  2024-09  stg_lic_2024_09      True  codigoexterno     sector                True                      True                 True                 
1  2024-10  stg_lic_2024_10      True  codigoexterno     sector                True                      True                 True                 
2  2024-11  stg_lic_2024_11      True  codigoexterno     sector                True                      True                 True                 
------------------------------------------------------------
== D6_control_meses_oc ==
shape: (14, 9)
cols: ['periodo', 'tabla', 'incluida', 'tiene_codigo', 'tiene_codigolicitacion', 'tiene_tipomonedaoc', '

## 7 — Conclusión metodológica

- Las métricas son agregadas y diagnósticas (DT_TEND/DT_SECT), sin vínculo transaccional 1:1.
- Se respeta segmentación por moneda **sin conversión**.
- Los CSV exportados constituyen evidencia reproducible para consumo en Power BI.
